# v5 + supplemental ICON-Global (raw): controlled October comparison + selector study (CPU)

Frozen pins: source `5aed997313b78678d11fa23b808c2a9835a1d91a`, archive SHA `ca160e76…eb980080`, bundle SHA `143b84e9…42125c`, adapter SHA `72f82ddc…9c953f` (13-test raw-default).
Core: v5 baseline (metadata base features) vs +7 raw ICON columns on identical 2025+ full-48h issue masks. Selector: lead×regime best-arm by valid-label September out-of-fit Brier (n≥50 else global fallback), applied to October as policy-transfer + frozen-pipeline variants, with a validation-frozen global-arm comparator; equal-blend reported too. R18 scored on its own population. Oct is exploratory, not pristine. Runs at lag 6 and lag 12 in separate fresh datasets — compare arms only within the same lag. Total ~15–25 min on Colab CPU, no GPU.

1. Run the setup cell and upload **both** files when prompted: `fra-v5-icon-source.zip` and `fra-v5-icon-colab-bundle.zip`.
2. Run the experiment cell. It executes the standalone script once per lag in a **fresh subprocess** (`--repo/--out/--lag` CLI), so no stale notebook imports leak in.
3. Download `/content/fra-v5-icon-results/lag6` and `lag12` (`metrics.csv` (120 cells), `per_row_traces.parquet`, `oof_selector_validation.parquet`, `selector_table.json`, `r18_events.json`, `r18_traces.parquet`, `manifest.json`, plus `pytest.txt`) and export this notebook before closing.

In [ ]:
import hashlib
import io
import os
import pathlib
import subprocess
import sys
import zipfile

REV = "5aed997313b78678d11fa23b808c2a9835a1d91a"
ARCHIVE_SHA = "ca160e76210755d6497e1120aed443664f343256b26d09ec4e50ba67eb980080"
BUNDLE_SHA = "143b84e94d07c62344137e14ba13fbedc436f15f5c8baa5c21ef0aa15142125c"
ADAPTER_SHA = "72f82ddc594ff07000f0caadcd95b645219faac0929f21d63ab08542cc9c953f"
ROOT = pathlib.Path("/content/fra-v5-icon")
OUT = pathlib.Path("/content/fra-v5-icon-results")

from google.colab import files
uploaded = files.upload()
for _name, _sha in (("fra-v5-icon-source.zip", ARCHIVE_SHA),
                      ("fra-v5-icon-colab-bundle.zip", BUNDLE_SHA)):
    assert _name in uploaded, f"upload {_name} when prompted"
    assert hashlib.sha256(uploaded[_name]).hexdigest() == _sha, f"{_name} SHA mismatch"
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(uploaded["fra-v5-icon-source.zip"])) as _z:
    _z.extractall(ROOT)
with zipfile.ZipFile(io.BytesIO(uploaded["fra-v5-icon-colab-bundle.zip"])) as _z:
    _z.extractall(ROOT)
assert hashlib.sha256((ROOT / "icon_supplement.py").read_bytes()).hexdigest() == ADAPTER_SHA
os.chdir(ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "ml/requirements.txt"], check=True)
OUT.mkdir(exist_ok=True)
result = subprocess.run([sys.executable, "-m", "pytest", "ml/tests", "-q"],
                        text=True, capture_output=True)
(OUT / "pytest.txt").write_text(result.stdout + result.stderr)
print(result.stdout, result.stderr)
assert result.returncode == 0, "Existing tests failed"
print("SETUP_AND_EXISTING_TESTS_PASSED", REV)


In [ ]:
import pathlib
import subprocess
import sys

ROOT = pathlib.Path("/content/fra-v5-icon")
OUT = pathlib.Path("/content/fra-v5-icon-results")
ARCHIVE_SHA = "ca160e76210755d6497e1120aed443664f343256b26d09ec4e50ba67eb980080"
ADAPTER_SHA = "72f82ddc594ff07000f0caadcd95b645219faac0929f21d63ab08542cc9c953f"
# Fresh subprocess per lag, separate datasets; compare arms only within the same lag.
# No notebook-cached imports (e.g. an older ml package) leak into either run.
for _lag in (6, 12):
    _out = OUT / f"lag{_lag}"
    _run = subprocess.run([sys.executable, str(ROOT / "experiment_v5_icon.py"),
                          "--repo", str(ROOT), "--out", str(_out), "--lag", str(_lag),
                          "--archive-sha", ARCHIVE_SHA, "--adapter-sha", ADAPTER_SHA])
    print(f"lag {_lag} returncode:", _run.returncode)
    assert _run.returncode == 0
print("BOTH_LAGS_COMPLETE")
